# 🧠 Brain Tumor Segmentation — Training Notebook
## ML Project by Rahul & Krishnaa for Dr. Valarmathi

**End-to-end**: Kaggle auth → Download → Train → Export → Deploy

| Dataset | Kaggle Slug | Cases | Volume |
|---------|-------------|-------|--------|
| BraTS 2020 | `awsaf49/brats20-dataset-training-validation` | 494 | 240×240×155 |
| BraTS 2021 | `dschettler8845/brats-2021-task1` | 1251 | 240×240×155 |
| BraTS 2023 | `shakilrana/brats-2023-adult-glioma` | ~1470 | 240×240×155 |
| BraTS 2024 | `i212385nomanarif/2024-brats-glioma` | ~1600 | 256×256×256 |

---

## 1. Environment Setup & GPU Check

In [ ]:
%%time
# ═══════════════════════════════════════════════════
#  INSTALL DEPENDENCIES + AUTO GPU SETUP
# ═══════════════════════════════════════════════════

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('📦 Installing dependencies...')
deps = ['torch', 'torchvision', 'nibabel', 'scipy', 'scikit-image',
        'scikit-learn', 'pandas', 'matplotlib', 'tqdm', 'xgboost',
        'kaggle', 'rich', 'psutil']
for d in deps:
    try:
        __import__(d.replace('-', '_'))
    except ImportError:
        install(d)

import os, sys, glob, json, time, random, warnings, gc, shutil
warnings.filterwarnings('ignore')
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from scipy import ndimage
from tqdm.notebook import tqdm
import psutil

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast  # Mixed precision

# ── GPU Setup ──
if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'\n🟢 GPU DETECTED: {gpu_name} ({gpu_mem:.1f} GB)')
    print(f'   CUDA version: {torch.version.cuda}')
    # Enable TF32 for Ampere+ GPUs (2x speed on A100/T4)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True  # Auto-tune convolution algorithms
    print(f'   TF32 enabled: {torch.backends.cuda.matmul.allow_tf32}')
    print(f'   cuDNN benchmark: {torch.backends.cudnn.benchmark}')
    USE_AMP = True  # Mixed precision training
    print(f'   Mixed precision (FP16): ON')
else:
    device = torch.device('cpu')
    print('\n🔴 NO GPU — Running on CPU (SLOW)')
    print('   Go to Runtime → Change runtime type → GPU')
    USE_AMP = False

# ── System info ──
ram = psutil.virtual_memory()
print(f'\n💾 RAM: {ram.total/1e9:.1f} GB total, {ram.available/1e9:.1f} GB free')
print(f'💿 Disk: {shutil.disk_usage("/").free/1e9:.1f} GB free')
print(f'🐍 Python: {sys.version.split()[0]}')
print(f'🔥 PyTorch: {torch.__version__}')
print(f'\n✅ Environment ready')

## 2. Upload Kaggle Credentials
Download `kaggle.json` from https://www.kaggle.com/settings → **API** → **Create New Token**

In [ ]:
from google.colab import files

print('📂 Upload your kaggle.json:')
uploaded = files.upload()

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

# Verify
ret = os.system('kaggle datasets list --sort-by votes --max-size 1 > /dev/null 2>&1')
if ret == 0:
    print('✅ Kaggle API authenticated successfully')
else:
    print('❌ Kaggle auth failed — check your kaggle.json')

## 3. Download BraTS Dataset

In [ ]:
#@title Choose dataset {display-mode: "form"}
PREFERRED = 'brats2020' #@param ['brats2020', 'brats2021', 'brats2023', 'brats2024']
DATA_DIR = '/content/brats_data'
os.makedirs(DATA_DIR, exist_ok=True)

ALL_DATASETS = {
    'brats2020': [
        ('awsaf49/brats20-dataset-training-validation', 'BraTS 2020 Train+Val — 494 cases, ~7.5 GB'),
        ('awsaf49/brats2020-training-data',             'BraTS 2020 Train Only — 369 cases, ~5.5 GB'),
    ],
    'brats2021': [('dschettler8845/brats-2021-task1', 'BraTS 2021 — 1251 cases, ~15 GB')],
    'brats2023': [('shakilrana/brats-2023-adult-glioma', 'BraTS 2023 Adult Glioma — ~1470 cases')],
    'brats2024': [
        ('i212385nomanarif/2024-brats-glioma',       'BraTS 2024 GLI Full — ~1600 cases'),
        ('nguyenthanhkhanh/brats2024-small-dataset',  'BraTS 2024 Small Subset'),
    ],
}

# Build attempt order
attempt_order = list(ALL_DATASETS.get(PREFERRED, []))
for ver in ['brats2020', 'brats2021', 'brats2023', 'brats2024']:
    for item in ALL_DATASETS[ver]:
        if item not in attempt_order:
            attempt_order.append(item)

downloaded_version = None
download_start = time.time()

for idx, (slug, desc) in enumerate(attempt_order):
    print(f'\n[{idx+1}/{len(attempt_order)}] 🔄 {desc}')
    print(f'    kaggle datasets download -d {slug}')
    t0 = time.time()
    ret = os.system(f'kaggle datasets download -d {slug} -p {DATA_DIR} --unzip 2>&1')
    elapsed = time.time() - t0

    if ret == 0:
        nii_count = len(list(Path(DATA_DIR).rglob('*.nii*')))
        if nii_count > 0:
            dl_speed = ''
            try:
                total_bytes = sum(f.stat().st_size for f in Path(DATA_DIR).rglob('*') if f.is_file())
                dl_speed = f' ({total_bytes/1e9:.1f} GB in {elapsed:.0f}s = {total_bytes/1e6/elapsed:.0f} MB/s)'
            except: pass
            print(f'    ✅ Downloaded! {nii_count} NIfTI files{dl_speed}')
            if '2024' in slug: downloaded_version = 'brats2024'
            elif '2023' in slug: downloaded_version = 'brats2023'
            elif '2021' in slug: downloaded_version = 'brats2021'
            else: downloaded_version = 'brats2020'
            break
        else:
            print(f'    ⚠️ No NIfTI files — trying next')
    else:
        print(f'    ❌ Failed ({elapsed:.0f}s) — have you accepted terms on kaggle.com?')

total_dl = time.time() - download_start
if downloaded_version:
    print(f'\n{"="*60}')
    print(f'  📊 Dataset: {downloaded_version}')
    print(f'  ⏱  Total download time: {timedelta(seconds=int(total_dl))}')
    !du -sh {DATA_DIR}
    print(f'{"="*60}')
else:
    raise RuntimeError('All downloads failed — accept dataset terms on kaggle.com first')

## 4. Discover & Validate Cases

In [ ]:
def detect_modalities(case_dir):
    """Auto-detect modality files — handles BraTS 2020/2021/2023/2024 naming."""
    d = Path(case_dir)
    niftis = sorted(d.glob('*.nii*'))
    r = {'t1': None, 't1ce': None, 't2': None, 'flair': None, 'seg': None}
    for f in niftis:
        n = f.name.lower()
        if 'seg' in n:
            r['seg'] = str(f)
        elif any(n.endswith(x) for x in ['_t1ce.nii.gz','_t1ce.nii','-t1c.nii.gz','-t1c.nii']):
            r['t1ce'] = str(f)
        elif any(n.endswith(x) for x in ['_flair.nii.gz','_flair.nii','-t2f.nii.gz','-t2f.nii']):
            r['flair'] = str(f)
        elif any(n.endswith(x) for x in ['_t1.nii.gz','_t1.nii','-t1n.nii.gz','-t1n.nii']):
            r['t1'] = str(f)
        elif any(n.endswith(x) for x in ['_t2.nii.gz','_t2.nii','-t2w.nii.gz','-t2w.nii']):
            r['t2'] = str(f)
    return r


print('🔍 Scanning for BraTS cases...')
scan_start = time.time()

all_niftis = list(Path(DATA_DIR).rglob('*.nii*'))
print(f'   Found {len(all_niftis)} total NIfTI files')

seen = set()
all_cases = []
pbar = tqdm(sorted(all_niftis), desc='Validating cases', unit='file')
for nii in pbar:
    if 'seg' in nii.name.lower():
        d = nii.parent
        if d not in seen:
            seen.add(d)
            mods = detect_modalities(d)
            n_mods = sum(1 for k in ['t1','t1ce','t2','flair'] if mods[k])
            if n_mods >= 1:
                all_cases.append(str(d))
    pbar.set_postfix(valid=len(all_cases))
all_cases.sort()

scan_time = time.time() - scan_start
print(f'\n✅ {len(all_cases)} valid cases found in {scan_time:.1f}s')

# ── Inspect first case ──
if all_cases:
    sample_dir = all_cases[0]
    mods = detect_modalities(sample_dir)
    print(f'\n📁 Sample: {Path(sample_dir).name}')
    for k, v in mods.items():
        st = '✅' if v else '❌'
        fn = Path(v).name if v else 'missing'
        print(f'   {st} {k:6s} → {fn}')

    first_mod = next(v for v in mods.values() if v and 'seg' not in v)
    vol = nib.load(first_mod).get_fdata()
    print(f'\n   Volume: {vol.shape}, range [{vol.min():.0f}, {vol.max():.0f}], non-zero: {(vol>0).mean()*100:.1f}%')

    if mods['seg']:
        seg = nib.load(mods['seg']).get_fdata()
        labels, counts = np.unique(seg, return_counts=True)
        print(f'   Labels: {labels.astype(int).tolist()}')
        for l, c in zip(labels, counts):
            if l > 0:
                bar = '█' * int(c / seg.size * 500)
                print(f'   Label {int(l):d}: {c:>10,} voxels ({c/seg.size*100:.2f}%) {bar}')

In [ ]:
from sklearn.model_selection import train_test_split

train_dirs, val_dirs = train_test_split(all_cases, test_size=0.15, random_state=42)
print(f'📊 Split: {len(train_dirs)} train / {len(val_dirs)} val')

## 5. Dataset & DataLoader

In [ ]:
class BraTSDataset(Dataset):
    """Universal BraTS loader — 2020/2021/2023/2024."""
    def __init__(self, case_dirs, target=(128,128,128), augment=False):
        self.cases = case_dirs; self.target = target; self.augment = augment
    def __len__(self): return len(self.cases)

    def _load(self, p): return nib.load(p).get_fdata().astype(np.float32)

    def _norm(self, v):
        m = v > 0
        if m.sum(): v[m] = (v[m]-v[m].mean())/(v[m].std()+1e-8)
        return v

    def _crop(self, v, t):
        o = np.zeros(t, dtype=v.dtype)
        ss, sd = [], []
        for i in range(3):
            s, g = v.shape[i], t[i]
            if s >= g: start=(s-g)//2; ss.append(slice(start,start+g)); sd.append(slice(0,g))
            else: pad=(g-s)//2; ss.append(slice(0,s)); sd.append(slice(pad,pad+s))
        o[sd[0],sd[1],sd[2]] = v[ss[0],ss[1],ss[2]]
        return o

    def _aug(self, v, s):
        for ax in range(3):
            if random.random()>0.5: v=np.flip(v,ax+1).copy(); s=np.flip(s,ax).copy()
        for c in range(v.shape[0]):
            m = v[c]!=0
            if m.sum(): v[c][m]*=np.random.uniform(0.9,1.1); v[c][m]+=np.random.uniform(-0.1,0.1)
        return v, s

    def __getitem__(self, idx):
        mods = detect_modalities(self.cases[idx])
        chs = []
        for k in ['t1','t1ce','t2','flair']:
            if mods[k] and os.path.exists(mods[k]):
                chs.append(self._crop(self._norm(self._load(mods[k])), self.target))
            else:
                chs.append(np.zeros(self.target, dtype=np.float32))
        seg = self._crop(self._load(mods['seg']).astype(np.int64), self.target).astype(np.int64) if mods['seg'] else np.zeros(self.target,dtype=np.int64)
        sm = np.zeros_like(seg); sm[seg==1]=1; sm[seg==2]=2; sm[seg==4]=3
        vol = np.stack(chs, 0)
        if self.augment: vol, sm = self._aug(vol, sm)
        return torch.FloatTensor(vol), torch.LongTensor(sm)

# Build loaders
BATCH = 2 if (torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_mem > 14e9) else 1
WORKERS = 4 if torch.cuda.is_available() else 2

train_ds = BraTSDataset(train_dirs, augment=True)
val_ds   = BraTSDataset(val_dirs)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=WORKERS, pin_memory=True, persistent_workers=True)
val_dl   = DataLoader(val_ds,   batch_size=1,     shuffle=False, num_workers=WORKERS, pin_memory=True, persistent_workers=True)

print(f'Batch size: {BATCH}, Workers: {WORKERS}')
print(f'Train: {len(train_ds)} cases ({len(train_dl)} batches)')
print(f'Val:   {len(val_ds)} cases')

# Test load one case with timer
print('\n⏳ Loading first case to verify...')
t0 = time.time()
sv, ss = train_ds[0]
load_time = time.time() - t0
print(f'   Loaded in {load_time:.1f}s — shape: {sv.shape}, labels: {torch.unique(ss).numpy()}')
print(f'   Estimated full epoch load time: ~{load_time * len(train_ds) / 60:.0f} min')

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
mid = sv.shape[1]//2
for i,(ax,t) in enumerate(zip(axes[:4],['T1','T1CE','T2','FLAIR'])):
    ax.imshow(sv[i,mid],cmap='gray'); ax.set_title(t,fontweight='bold'); ax.axis('off')
axes[4].imshow(ss[mid],cmap='nipy_spectral',vmin=0,vmax=3)
axes[4].set_title('GT',fontweight='bold'); axes[4].axis('off')
plt.suptitle(f'Sample — Slice {mid}',fontweight='bold'); plt.tight_layout(); plt.show()

## 6. 3D U-Net Model

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, ic, oc, drop=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(ic,oc,3,padding=1), nn.InstanceNorm3d(oc), nn.LeakyReLU(0.01,True), nn.Dropout3d(drop),
            nn.Conv3d(oc,oc,3,padding=1), nn.InstanceNorm3d(oc), nn.LeakyReLU(0.01,True))
    def forward(self,x): return self.net(x)

class UNet3D(nn.Module):
    def __init__(self, in_ch=4, out_ch=4, feats=[32,64,128,256]):
        super().__init__()
        self.encs=nn.ModuleList(); self.decs=nn.ModuleList()
        self.pools=nn.ModuleList(); self.ups=nn.ModuleList()
        ch=in_ch
        for f in feats:
            self.encs.append(ConvBlock(ch,f)); self.pools.append(nn.MaxPool3d(2)); ch=f
        self.bottleneck = ConvBlock(feats[-1], feats[-1]*2, drop=0.2)
        for f in reversed(feats):
            self.ups.append(nn.ConvTranspose3d(f*2,f,2,stride=2))
            self.decs.append(ConvBlock(f*2,f))
        self.head = nn.Conv3d(feats[0],out_ch,1)

    def forward(self,x):
        skips=[]
        for enc,pool in zip(self.encs,self.pools): x=enc(x); skips.append(x); x=pool(x)
        x=self.bottleneck(x)
        for up,dec,sk in zip(self.ups,self.decs,reversed(skips)):
            x=up(x)
            d=[s-x_ for s,x_ in zip(sk.shape[2:],x.shape[2:])]
            if any(dd!=0 for dd in d): x=F.pad(x,[0,d[2],0,d[1],0,d[0]])
            x=torch.cat([sk,x],1); x=dec(x)
        return self.head(x)

model = UNet3D().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'🧬 Model: UNet3D')
print(f'   Parameters: {n_params:,} ({n_params*4/1e6:.1f} MB)')
print(f'   Device: {device}')

# Quick GPU memory check
if torch.cuda.is_available():
    dummy = torch.randn(1, 4, 128, 128, 128, device=device)
    with torch.no_grad():
        _ = model(dummy)
    mem_used = torch.cuda.max_memory_allocated() / 1e9
    mem_total = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'   GPU memory: {mem_used:.1f} / {mem_total:.1f} GB (single inference)')
    del dummy; torch.cuda.empty_cache(); gc.collect()
    print(f'   ✅ Model fits in GPU memory')

## 7. Loss & Metrics

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.): super().__init__(); self.s=smooth
    def forward(self, p, t):
        ps=F.softmax(p,1); toh=F.one_hot(t,p.shape[1]).permute(0,4,1,2,3).float()
        i=(ps*toh).sum(dim=(2,3,4)); u=ps.sum(dim=(2,3,4))+toh.sum(dim=(2,3,4))
        return 1-(2*i+self.s)/(u+self.s)[:,1:].mean()

class ComboLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice=DiceLoss()
        self.ce=nn.CrossEntropyLoss(weight=torch.FloatTensor([0.1,1.,0.8,1.2]).to(device))
    def forward(self,p,t): return .5*self.dice(p,t)+.5*self.ce(p,t)

def dice_scores(pred,target):
    s={}
    for c,name in [(1,'NCR'),(2,'ED'),(3,'ET')]:
        pc,tc = pred==c, target==c
        s[name]=float(2*(pc&tc).sum()/(pc.sum()+tc.sum()+1e-8))
    wtp,wtt=pred>0,target>0; s['WT']=float(2*(wtp&wtt).sum()/(wtp.sum()+wtt.sum()+1e-8))
    tcp,tct=(pred==1)|(pred==3),(target==1)|(target==3); s['TC']=float(2*(tcp&tct).sum()/(tcp.sum()+tct.sum()+1e-8))
    return s

criterion = ComboLoss()
print('✅ Loss: Dice + CrossEntropy (class-weighted)')

## 8. Training Loop
Uses **mixed-precision (FP16)** on GPU for ~2× speedup + lower memory.

In [ ]:
#@title Training Config {display-mode: "form"}
EPOCHS = 50    #@param {type: "integer"}
LR = 1e-4      #@param {type: "number"}
PATIENCE = 10  #@param {type: "integer"}
CKPT = '/content/checkpoints'; os.makedirs(CKPT, exist_ok=True)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = GradScaler(enabled=USE_AMP)

hist = {'tl':[],'vl':[],'wt':[],'tc':[],'et':[],'lr':[]}
best_dice = 0.; wait = 0
train_start = time.time()

print(f'\n{"═"*70}')
print(f'  🏋️ TRAINING: {EPOCHS} epochs, LR={LR}, batch={BATCH}')
print(f'  GPU: {"ON (FP16 mixed precision)" if USE_AMP else "OFF (CPU, will be slow)"}')
print(f'  Estimated time: ~{len(train_dl)*15/60:.0f} min/epoch on T4 GPU')
print(f'{"═"*70}\n')

for epoch in range(EPOCHS):
    ep_start = time.time()

    # ── Train ──
    model.train()
    t_loss = 0
    pbar = tqdm(train_dl, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]',
                bar_format='{l_bar}{bar:30}{r_bar}', leave=False)
    for batch_i, (vol, seg) in enumerate(pbar):
        vol, seg = vol.to(device, non_blocking=True), seg.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=USE_AMP):
            out = model(vol)
            loss = criterion(out, seg)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        t_loss += loss.item()

        # Live stats
        gpu_mem = f'{torch.cuda.memory_allocated()/1e9:.1f}G' if torch.cuda.is_available() else 'CPU'
        elapsed_ep = time.time() - ep_start
        eta_ep = elapsed_ep / (batch_i+1) * (len(train_dl)-batch_i-1)
        pbar.set_postfix(loss=f'{loss.item():.4f}', gpu=gpu_mem, eta=f'{eta_ep:.0f}s')
    t_loss /= len(train_dl)

    # ── Validate ──
    model.eval()
    v_loss = 0; all_d = []
    pbar_v = tqdm(val_dl, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]',
                  bar_format='{l_bar}{bar:30}{r_bar}', leave=False)
    with torch.no_grad():
        for vol, seg in pbar_v:
            vol, seg = vol.to(device, non_blocking=True), seg.to(device, non_blocking=True)
            with autocast(enabled=USE_AMP):
                out = model(vol)
                v_loss += criterion(out, seg).item()
            pred = torch.argmax(out, 1).cpu().numpy()
            for i in range(pred.shape[0]):
                all_d.append(dice_scores(pred[i], seg[i].cpu().numpy()))
    v_loss /= len(val_dl)

    avg = {k: np.mean([d[k] for d in all_d]) for k in all_d[0]}
    mean_d = (avg['WT']+avg['TC']+avg['ET'])/3
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    hist['tl'].append(t_loss); hist['vl'].append(v_loss)
    hist['wt'].append(avg['WT']); hist['tc'].append(avg['TC']); hist['et'].append(avg['ET'])
    hist['lr'].append(lr)

    ep_time = time.time() - ep_start
    total_elapsed = time.time() - train_start
    remaining = ep_time * (EPOCHS - epoch - 1)

    # ── Pretty status line ──
    save_str = ''
    if mean_d > best_dice:
        best_dice = mean_d; wait = 0
        torch.save({'epoch':epoch,'model':model.state_dict(),'dice':best_dice,'hist':hist},
                   f'{CKPT}/best.pth')
        save_str = ' ✅ BEST'
    else:
        wait += 1

    prog = '█' * int((epoch+1)/EPOCHS*20) + '░' * (20-int((epoch+1)/EPOCHS*20))
    print(
        f'E{epoch+1:3d} [{prog}] '
        f'TL:{t_loss:.4f} VL:{v_loss:.4f} │ '
        f'WT:{avg["WT"]:.3f} TC:{avg["TC"]:.3f} ET:{avg["ET"]:.3f} │ '
        f'Mean:{mean_d:.4f} │ '
        f'{ep_time:.0f}s │ ETA:{timedelta(seconds=int(remaining))}'
        f'{save_str}'
    )

    if wait >= PATIENCE:
        print(f'\n⛔ Early stopping — no improvement for {PATIENCE} epochs')
        break

total_time = time.time() - train_start
print(f'\n{"═"*70}')
print(f'  🏁 Training complete!')
print(f'  Best dice: {best_dice:.4f}')
print(f'  Total time: {timedelta(seconds=int(total_time))}')
print(f'  Epochs: {len(hist["tl"])}')
if torch.cuda.is_available():
    print(f'  Peak GPU: {torch.cuda.max_memory_allocated()/1e9:.1f} GB')
print(f'{"═"*70}')

## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(hist['tl'],label='Train',lw=2); axes[0].plot(hist['vl'],label='Val',lw=2)
axes[0].set_title('Loss',fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(hist['wt'],label='WT',lw=2); axes[1].plot(hist['tc'],label='TC',lw=2); axes[1].plot(hist['et'],label='ET',lw=2)
axes[1].set_title('Val Dice',fontweight='bold'); axes[1].legend(); axes[1].set_ylim([0,1]); axes[1].grid(alpha=.3)
axes[2].plot(hist['lr'],color='green',lw=2); axes[2].set_title('LR Schedule',fontweight='bold'); axes[2].grid(alpha=.3)
plt.tight_layout(); plt.savefig('/content/curves.png',dpi=150); plt.show()

## 10. Inference & Visualization

In [ ]:
ckpt = torch.load(f'{CKPT}/best.pth', map_location=device)
model.load_state_dict(ckpt['model']); model.eval()
print(f'Loaded best from epoch {ckpt["epoch"]+1} (dice={ckpt["dice"]:.4f})')

N_VIS = min(5, len(val_ds))
for ci in tqdm(range(N_VIS), desc='Running inference'):
    v, s = val_ds[ci]
    t0 = time.time()
    with torch.no_grad(), autocast(enabled=USE_AMP):
        pred = torch.argmax(model(v.unsqueeze(0).to(device)), 1).squeeze().cpu().numpy()
    inf_time = time.time() - t0
    s = s.numpy()
    d = dice_scores(pred, s)

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for row, sl in enumerate([pred.shape[0]//3, pred.shape[0]//2, 2*pred.shape[0]//3]):
        axes[row,0].imshow(v[1,sl],cmap='gray'); axes[row,0].set_title(f'T1CE z={sl}')
        axes[row,1].imshow(s[sl],cmap='nipy_spectral',vmin=0,vmax=3); axes[row,1].set_title('GT')
        axes[row,2].imshow(pred[sl],cmap='nipy_spectral',vmin=0,vmax=3); axes[row,2].set_title('Pred')
        axes[row,3].imshow(v[1,sl],cmap='gray'); axes[row,3].imshow(pred[sl],cmap='nipy_spectral',alpha=.4,vmin=0,vmax=3)
        axes[row,3].set_title('Overlay')
        for ax in axes[row]: ax.axis('off')
    plt.suptitle(f'Case {ci} │ WT:{d["WT"]:.3f} TC:{d["TC"]:.3f} ET:{d["ET"]:.3f} │ {inf_time:.2f}s', fontweight='bold')
    plt.tight_layout(); plt.show()

## 11. Train Grading Classifier

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
import pickle

def grade_feats(seg):
    t=(seg>0).sum()
    if t==0: return None
    et,ncr,ed=(seg==3).sum(),(seg==1).sum(),(seg==2).sum()
    return [t/1000, et/t, ncr/t, ed/t, et/(ncr+1)]

def assign_grade(f):
    if f[1]>.35 and f[2]>.15: return 3
    elif f[1]>.2: return 2
    elif f[0]>15: return 1
    return 0

X, y = [], []
for i in tqdm(range(len(train_ds)), desc='Extracting grading features'):
    _, seg = train_ds[i]
    f = grade_feats(seg.numpy())
    if f: X.append(f); y.append(assign_grade(f))
X, y = np.array(X), np.array(y)

print(f'\nSamples: {len(X)}, Grade distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

clf = XGBClassifier(n_estimators=100, max_depth=5, random_state=42, eval_metric='mlogloss',
                    tree_method='gpu_hist' if torch.cuda.is_available() else 'hist')
cv = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
print(f'CV accuracy: {cv.mean():.4f} ± {cv.std():.4f}')
clf.fit(X, y)
with open(f'{CKPT}/grading.pkl', 'wb') as f: pickle.dump(clf, f)
print('✅ Grading classifier saved')

## 12. Export for Backend Deployment

In [ ]:
EXP = '/content/export'; os.makedirs(EXP, exist_ok=True)

print('📦 Exporting...')
tasks = [
    ('Model weights', lambda: torch.save(model.state_dict(), f'{EXP}/unet3d_brats.pth')),
    ('Grading classifier', lambda: shutil.copy(f'{CKPT}/grading.pkl', f'{EXP}/grading_classifier.pkl')),
    ('Model card', lambda: json.dump({
        'model':'UNet3D','dataset':downloaded_version,'best_dice':float(best_dice),
        'metrics':{'WT':hist['wt'][-1],'TC':hist['tc'][-1],'ET':hist['et'][-1]},
        'grading_cv':float(cv.mean()),
        'training':{'epochs':len(hist['tl']),'lr':LR,'loss':'Dice+CE','amp':USE_AMP},
        'total_cases':len(all_cases),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    }, open(f'{EXP}/model_card.json','w'), indent=2)),
    ('Dataset manifest', lambda: json.dump({
        'version':downloaded_version,'total':len(all_cases),
        'train':len(train_dirs),'val':len(val_dirs),
    }, open(f'{EXP}/dataset_manifest.json','w'), indent=2)),
    ('Training history', lambda: pd.DataFrame(hist).to_csv(f'{EXP}/history.csv', index=False)),
]

for name, fn in tqdm(tasks, desc='Exporting artifacts'):
    fn()

print(f'\n📦 Exported to {EXP}/')
for f in sorted(os.listdir(EXP)):
    sz = os.path.getsize(f'{EXP}/{f}')
    unit = 'MB' if sz > 1e6 else 'KB'
    val = sz/1e6 if sz > 1e6 else sz/1e3
    print(f'   {f:40s} {val:>8.1f} {unit}')

print(f'\n🚀 Deploy:')
print(f'   cp {EXP}/unet3d_brats.pth backend/models/')
print(f'   cp {EXP}/grading_classifier.pkl backend/models/')
print(f'   cd backend && uvicorn app.main:app --reload')

## 13. Save to Google Drive

In [ ]:
try:
    from google.colab import drive, files as colab_files
    print('📁 Mounting Google Drive...')
    drive.mount('/content/drive')
    GDRIVE = '/content/drive/MyDrive/ML_BrainTumor'
    
    tasks = [
        ('Export files', lambda: shutil.copytree(EXP, GDRIVE, dirs_exist_ok=True)),
        ('Training curves', lambda: shutil.copy('/content/curves.png', GDRIVE)),
    ]
    for name, fn in tqdm(tasks, desc='Saving to Drive'):
        fn()
    
    print(f'\n✅ Saved to: {GDRIVE}')
    for f in sorted(os.listdir(GDRIVE)):
        sz = os.path.getsize(f'{GDRIVE}/{f}')/1e6
        print(f'   {f}: {sz:.1f} MB')

except Exception as e:
    print(f'⚠️ Drive not available ({e})')
    print('Downloading files directly...')
    from google.colab import files as colab_files
    for f in ['unet3d_brats.pth', 'grading_classifier.pkl', 'model_card.json']:
        try:
            colab_files.download(f'{EXP}/{f}')
            print(f'   📥 {f}')
        except: pass

---
## ✅ Done!

**Deploy to backend:**
```bash
cp export/unet3d_brats.pth backend/models/
cp export/grading_classifier.pkl backend/models/
cd backend && uvicorn app.main:app --reload --port 8000
```

**Frontend:** Open `frontend/index.jsx` → connects to `localhost:8000` automatically.

---
*Rahul & Krishnaa • Dr. Valarmathi Lab • VIT Chennai*